# CUDA fundamentals — coding quiz

Work from first principles. Attempt each section before reviewing pmpp.ipynb. Checks provide feedback, not solutions.

Focus: tensor layout, loops, row-major indexing, broadcasting, launch geometry, and the Python → C++/CUDA boundary.

In [8]:
import math
import torch

torch.manual_seed(7)

## 1. Image layout — fill the blanks

An RGB image has shape (3, h, w): channels first. After flattening contiguous storage, all red pixels appear first, then green, then blue. Complete the one-pixel grayscale calculation.

In [9]:
def rgb2grey_one_pixel(x, i):
    # x has shape (3, h, w); i is a flat spatial-pixel index
    _, h, w = x.shape
    n = h*w
    flat = x.flatten()
    # print(flat.shape)
    return 0.2989 * flat[i] + 0.5870 * flat[i+n] + 0.1140 * flat[i+2*n]

tiny = torch.tensor([[[10, 20], [30, 40]],
                     [[50, 60], [70, 80]],
                     [[90, 100], [110, 120]]], dtype=torch.float32)
expected_pixel_2 = torch.tensor(0.2989*30 + 0.5870*70 + 0.1140*110)
assert torch.isclose(rgb2grey_one_pixel(tiny, 2), expected_pixel_2)

## 2. Write the grayscale loop from scratch

Allocate one output value per spatial pixel. Use a single flat loop, then reshape to (h, w). Do not use broadcasting, .mean, or matrix multiplication.

In [10]:
def rgb2grey_loop(x):
    # YOUR CODE
    c, h, w = x.shape
    n = h*w
    print(x.shape)
    x = x.flatten()
    print(x.shape)
    res = torch.empty(n)
    for i in range(n): res[i] = 0.2989 * x[i] + 0.5870 * x[i+n] + 0.1140 * x[i+2*n]
    return res.view(h, w)

expected = 0.2989*tiny[0] + 0.5870*tiny[1] + 0.1140*tiny[2]
assert torch.allclose(rgb2grey_loop(tiny), expected)
assert rgb2grey_loop(tiny).shape == (2, 2)

torch.Size([3, 2, 2])
torch.Size([12])
torch.Size([3, 2, 2])
torch.Size([12])


## 3. Kernel logic versus launch logic

This is intentionally a serial Python simulator. Write a kernel that handles one logical index, then a launcher that invokes it n times. This separation prepares you for CUDA; it is not CPU parallelism.

In [11]:
def grey_kernel(i, flat_x, out, n):
    # YOUR CODE: write exactly out[i]
    out[i] = 0.2989 * flat_x[i] + 0.5870 * flat_x[i+n] + 0.1140 * flat_x[i+2*n]
    pass

def run_kernel(kernel, n, *args):
    # YOUR CODE: serial simulator of n logical threads
    for i in range(n): kernel(i, *args)
    pass

def rgb2grey_kernel_style(x):
    _, h, w = x.shape
    n = h * w
    out = torch.empty(n, dtype=x.dtype, device=x.device)
    run_kernel(grey_kernel, n, x.flatten(), out, n)
    return out.view(h, w)

assert torch.allclose(rgb2grey_kernel_style(tiny), expected)

## 4. Blocks, threads, and bounds

Fill the launch math. For n=1000 and threads=256, exactly 4 blocks launch. The final block covers indices 768–1023, but only 768–999 are valid.

In [12]:
def ceil_div(a, b):
    return int(math.ceil(a/b))

def global_index(block_idx, thread_idx, block_dim):
    return block_idx * block_dim + thread_idx

assert ceil_div(1000, 256) == 4
assert global_index(3, 0, 256) == 768
assert global_index(3, 255, 256) == 1023

def grey_block_kernel(block_idx, thread_idx, block_dim, flat_x, out, n):
    i = global_index(block_idx, thread_idx, block_dim)
    # YOUR CODE: guard invalid i, then write out[i]
    if (i<n): out[i] = 0.2989 * flat_x[i] + 0.5870 * flat_x[i+n] + 0.1140 * flat_x[i+2*n]
    pass

### Explain in your own words

Why does ceiling division deliberately create excess logical threads, and why is the i < n guard required for correctness?

threads are cheap in gpu, if we have 1:1 mapping of threads to data elements, we need at least that many threads. therefore, when we run the kernel we need to guard against the executions of threads for which there is no data elements

## 5. Matrix multiplication — construct the loops

For A[h, k] @ B[k, w], output is C[h, w]. Each C[r, c] is the dot product of row r in A and column c in B. Write the three nested loops. Avoid @, matmul, .mm, einsum, broadcasting, and .sum.

In [13]:
def matmul_loops(a, b):
    ar, ac = a.shape
    br, bc = b.shape
    print((ar, ac), (br, bc))
    t_out = torch.zeros(ar, bc)
    print(t_out)
    for i in range(ar): # 2
        for j in range(bc): # 4
            for k in range(ac): # 3
                t_out[i, j] += a[i, k] * b[k,j]

    return t_out
        

a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # (h=2, k=3)
b = torch.tensor([[1., 2., 3., 4.],
                  [5., 6., 7., 8.],
                  [9., 10., 11., 12.]])          # (k=3, w=4)
assert torch.equal(matmul_loops(a, b), a @ b)
assert matmul_loops(a, b).shape == (2, 4)

(2, 3) (3, 4)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])
(2, 3) (3, 4)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])


## 6. Broadcasting — build it one transformation at a time

For fixed row i, a[i, :] has shape (k,). Add a singleton axis to make it (k, 1); multiplying by b[k, w] broadcasts it to (k, w). Summing dimension 0 gives (w,). Fill each expression, then inspect shapes.

In [14]:
i = 0
row = a[i, :]                 # shape: (k,)
# print(a)
print(row)
column = a[i, :, None]            # shape: (k, 1)
print(column)
products = column * b               # shape: (k, w)
out_row = products.sum(dim=0)  # shape: (w,)

assert row.shape == (3,)
assert column.shape == (3, 1)
assert products.shape == (3, 4)
assert out_row.shape == (4,)
assert torch.equal(out_row, (a @ b)[i])

tensor([1., 2., 3.])
tensor([[1.],
        [2.],
        [3.]])


### 6b. Turn one broadcast row into full matmul

Use exactly one Python loop over output rows. Inside it, use the previous broadcasting expression to create each complete output row. Explain why this is generally faster on CPU than three Python loops, despite equivalent math.

In [15]:
def matmul_broadcast_rows(a, b):
    # YOUR CODE: one Python loop; vectorized work inside the loop
    ar, ac = a.shape
    br, bc = b.shape
    t_out = torch.zeros(ar, bc)
    for i in range(ar):
        column = a[i, :, None]
        print(f"\nOutput row {i}: A row {i} reshaped into a column, shape {tuple(column.shape)}. Each value will scale the matching row of B:\n{column}")
        products = column * b
        print(f"Row {i}: element-by-element products after broadcasting against B, shape {tuple(products.shape)}. Each column holds the contributions to one output value:\n{products}")
        out_row = products.sum(dim=0)
        print(f"Row {i}: sum down the rows (dim=0), leaving one total per column, shape {tuple(out_row.shape)}:\n{out_row}")
        t_out[i, :] = out_row
        print(f"Output after storing row {i}. Rows 0 through {i} are complete; any later rows are still initial zeros:\n{t_out}")

    return t_out

assert torch.equal(matmul_broadcast_rows(a, b), a @ b)


Output row 0: A row 0 reshaped into a column, shape (3, 1). Each value will scale the matching row of B:
tensor([[1.],
        [2.],
        [3.]])
Row 0: element-by-element products after broadcasting against B, shape (3, 4). Each column holds the contributions to one output value:
tensor([[ 1.,  2.,  3.,  4.],
        [10., 12., 14., 16.],
        [27., 30., 33., 36.]])
Row 0: sum down the rows (dim=0), leaving one total per column, shape (4,):
tensor([38., 44., 50., 56.])
Output after storing row 0. Rows 0 through 0 are complete; any later rows are still initial zeros:
tensor([[38., 44., 50., 56.],
        [ 0.,  0.,  0.,  0.]])

Output row 1: A row 1 reshaped into a column, shape (3, 1). Each value will scale the matching row of B:
tensor([[4.],
        [5.],
        [6.]])
Row 1: element-by-element products after broadcasting against B, shape (3, 4). Each column holds the contributions to one output value:
tensor([[ 4.,  8., 12., 16.],
        [25., 30., 35., 40.],
        [54., 

_Why is this faster than the three-loop version? Your explanation here._

python does not direct every individual mul and add. it asks pytorch to do whole tensor operations in compiled code, which uses SIMD etc

## 7. 2-D CUDA indexing — translate the CPU loops

Assume tpb.x = tpb.y = 16. Fill global output row/column mapping and flat row-major offsets. Then state which output tile block (x=2, y=1) owns.

In [16]:
def matmul_thread_indices(block_x, block_y, thread_x, thread_y, block_dim_x=16, block_dim_y=16):
    c = block_x * block_dim_x + thread_x
    r = block_y * block_dim_y + thread_y

    return r, c

def flat_matmul_offsets(r, c, i, k, w):
    a_offset = r * k + i
    b_offset = i * w + c # B[i, c]
    out_offset = r * w + c     # out[r, c]
    return a_offset, b_offset, out_offset

assert matmul_thread_indices(2, 1, 0, 0) == (16, 32)
assert matmul_thread_indices(2, 1, 15, 15) == (31, 47)
assert flat_matmul_offsets(2, 3, 4, k=5, w=7) == (14, 31, 17)

## 8. Mastery challenge — design the CUDA wrapper and kernel

Without copying from the lecture, write pseudocode or valid C++/CUDA for a float32 matmul extension. Include:

1. CUDA and contiguous input checks.
2. Dimension extraction and size validation.
3. Output allocation preserving input options.
4. A 2-D 16×16 launch grid with ceiling division.
5. Kernel row/column indices, bounds guard, dot-product loop, and output store.
6. A post-launch CUDA error check.

Then explain why this correct naive kernel will usually lose to a @ b: identify at least two reasons.

In [52]:

a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # (h=2, k=3)
b = torch.tensor([[1., 2., 3., 4.],
                  [5., 6., 7., 8.],
                  [9., 10., 11., 12.]])    

In [80]:
def matmul_kernel(r, c, flat_a, flat_b, flat_out, k, w):
    total = 0
    for i in range(k):
        a_offset = r * k + i
        b_offset = i * w + c
        total += flat_a[a_offset] * flat_b[b_offset]
    
    flat_out[r*w+c] = total


In [65]:
def matmul_wrapper(a, b):
    ar, ac = a.shape
    br, bc = b.shape
    assert(ac == br)
    assert(a.dtype == b.dtype)
    assert(a.device == b.device)
    print(a.device)
    t_out = torch.zeros(ar, bc, dtype=a.dtype, device=a.device ).flatten()
    a = a.flatten()
    b = b.flatten()

    for i in range(ar):
        for j in range(bc):
            matmul_kernel(i, j, a, b, t_out, ac, bc)
    return t_out.view(ar, bc)

matmul_wrapper(a,b)
assert torch.allclose(matmul_wrapper(a, b), torch.matmul(a,b))

cpu
cpu


In [ ]:
def matmul_thread_indices(block_x, block_y, thread_x, thread_y, block_dim_x=16, block_dim_y=16):
    c = block_x * block_dim_x + thread_x
    r = block_y * block_dim_y + thread_y

    return r, c

In [79]:
from types import SimpleNamespace as ns

def matmul_launcher(a, b):
    h,k  = a.shape
    k2,w = b.shape
    assert k==k2, "Size mismatch!"
    output = torch.zeros(h, w, dtype=a.dtype, device=a.device).flatten()
    tpb = ns(x=16,y=16)
    blocks = ns(x=math.ceil(w/tpb.x), y=math.ceil(h/tpb.y))

    for i0 in range(blocks.x):
        for i1 in range(blocks.y):
            for j0 in range(tpb.x):
                for j1 in range(tpb.y):
                    r,c = matmul_thread_indices(i0, i1, j0, j1)
                    if (r<h and c<w):
                        matmul_kernel(r, c, a.flatten(), b.flatten(), output, k, w)
    
    # print(output.view(h,w))
    return output.view(h,w)

# matmul_launcher(a,b)
assert torch.allclose(matmul_launcher(a, b), torch.matmul(a,b))

c = torch.rand(17,3)
d = torch.rand(3, 35)
assert torch.allclose(matmul_launcher(c, d), torch.matmul(c,d))

In [ ]:
import statistics
import torch
from torch.utils.benchmark import Timer

# Keep this fixed across sizes. Change deliberately for a separate experiment.
CPU_THREADS = 1
BENCHMARK_SEED = 7

def benchmark_cpu_matmul(fn, sizes, label):
    results = []
    previous = None
    original_threads = torch.get_num_threads()
    torch.set_num_threads(CPU_THREADS)
    try:
        print(f"{label} | PyTorch {torch.__version__} | CPU threads={CPU_THREADS}")
        print("Times include output allocation; exclude input creation and validation.")
        print("Repeated inputs are reused: this measures steady-state performance.")
        print(f"{'N':>6} {'median ms':>12} {'IQR %':>9} {'blocks':>8} {'GFLOP/s':>10} {'time ratio':>12} {'work ratio':>12}")
        for size in sizes:
            generator = torch.Generator(device="cpu").manual_seed(BENCHMARK_SEED + size)
            a = torch.rand(size, size, generator=generator, dtype=torch.float32, device="cpu")
            b = torch.rand(size, size, generator=generator, dtype=torch.float32, device="cpu")
            # Validate outside timing. For the PyTorch baseline, these checks
            # verify its output contract; they are not an independent math test.
            output = fn(a, b)
            assert output.shape == (size, size)
            assert output.dtype == a.dtype and output.device == a.device
            if fn is not torch.matmul:
                torch.testing.assert_close(output, torch.matmul(a, b), rtol=1e-4, atol=1e-5)
            print(f"  N={size}: A={a.device}, B={b.device}, output={output.device}, dtype={output.dtype}")
            del output
            timer = Timer(stmt="fn(a, b)", globals={"fn": fn, "a": a, "b": b}, num_threads=CPU_THREADS)
            # Timer warms up and batches fast calls to reduce clock overhead.
            # Collect at least five blocks, even when one call exceeds 0.5 s.
            samples = []
            while len(samples) < 5:
                measurement = timer.blocked_autorange(min_run_time=0.5)
                samples.extend(measurement.times)  # seconds per call in each block
            median = statistics.median(samples)
            q1, _, q3 = statistics.quantiles(samples, n=4, method="inclusive")
            iqr_percent = 100 * (q3 - q1) / median
            gflops = 2 * size**3 / median / 1e9  # conventional approximate matmul FLOPs
            time_ratio = median / previous[1] if previous else None
            work_ratio = (size / previous[0])**3 if previous else None
            tr = f"{time_ratio:.2f}x" if previous else "--"
            wr = f"{work_ratio:.2f}x" if previous else "--"
            print(f"{size:6d} {median*1000:12.4f} {iqr_percent:9.1f} {len(samples):8d} {gflops:10.4f} {tr:>12} {wr:>12}")
            if iqr_percent > 10:
                print("  Timing variability is high; rerun while the machine is otherwise idle.")
            results.append(dict(size=size, median_s=median, iqr_percent=iqr_percent,
                                samples_s=samples, gflops=gflops, cpu_threads=CPU_THREADS))
            previous = (size, median)
    finally:
        torch.set_num_threads(original_threads)
    return results

# Run after the matmul_kernel / matmul_thread_indices / matmul_launcher cells.
python_cpu_results = benchmark_cpu_matmul(matmul_launcher, (32, 64), "Python launcher")


**Historical timings from the original three-run benchmark (not directly comparable to the fixed-thread results above).**

My version: 

Matrix size: 64 × 64
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 2.6602 seconds

Matrix size: 128 × 128
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 21.4702 seconds

In [ ]:
# Same timing method, dtype, device, seed, and CPU thread count as above.
# Large sizes are practical here because PyTorch uses compiled matrix operations.
pytorch_cpu_results = benchmark_cpu_matmul(
    torch.matmul, (32, 64, 128, 512, 1024, 2048, 4096, 8192), "PyTorch matmul"
)


**Historical timings from the original three-run benchmark (not directly comparable to the fixed-thread results above).**

Pytorch version:

Matrix size: 64 × 64
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 0.0007 seconds

Matrix size: 128 × 128
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 0.0002 seconds

Matrix size: 2048 × 2048
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 0.0939 seconds

Matrix size: 8192 × 8192
Devices: A=cpu, B=cpu, output=cpu
Average CPU time: 4.4973 seconds

### 8A. Translate the validated Python to CUDA

Agent-assisted translation of `matmul_kernel`, `matmul_thread_indices`, and `matmul_launcher`.
Run the cells below in order in the existing NGC GPU kernel. The Python implementation above is preserved.

| Python | CUDA translation |
|---|---|
| `matmul_thread_indices(...)` | `blockIdx`, `blockDim`, and `threadIdx` supply coordinates inside the kernel |
| Four simulated block/thread loops | One CUDA launch creates the grid |
| `if r < h and c < w` | Threads outside the output return before reading or writing memory |
| Loop over `i` and flat offsets | Same dot product, now using float pointers |
| Allocate output, return its 2-D view | Allocate a 2-D tensor; its contiguous storage is already flat in memory |

This is forward-only float32 matmul: no gradients, shared-memory tiling, or Tensor Core operations.
We use `empty` instead of `zeros`: every valid output is assigned once, including zero when k=0.
The input device guard and current stream keep this extension consistent with surrounding PyTorch operations.

References: [PyTorch inline extensions](https://docs.pytorch.org/docs/stable/cpp_extension.html),
[CUDA semantics and timing](https://docs.pytorch.org/docs/stable/notes/cuda.html).

In [96]:
import torch

# load_inline generates a Python binding for this host-side function.
cpp_source = r"""
torch::Tensor matmul_cuda(torch::Tensor a, torch::Tensor b);
"""

cuda_source = r"""
#include <torch/types.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAStream.h>
#include <c10/cuda/CUDAException.h>

// Your Python matmul_kernel + matmul_thread_indices.
__global__ void matmul_kernel_cuda(
    const float* flat_a, const float* flat_b, float* flat_out,
    int64_t h, int64_t k, int64_t w) {
    const int64_t r = int64_t(blockIdx.y) * blockDim.y + threadIdx.y;
    const int64_t c = int64_t(blockIdx.x) * blockDim.x + threadIdx.x;
    if (r >= h || c >= w) return;

    float total = 0.0f;
    for (int64_t i = 0; i < k; ++i) {
        const int64_t a_offset = r * k + i;
        const int64_t b_offset = i * w + c;
        total += flat_a[a_offset] * flat_b[b_offset];
    }
    flat_out[r * w + c] = total;
}

// Your Python matmul_launcher, with actual GPU execution.
torch::Tensor matmul_cuda(torch::Tensor a, torch::Tensor b) {
    TORCH_CHECK(a.is_cuda() && b.is_cuda(), "A and B must be CUDA tensors");
    TORCH_CHECK(a.device() == b.device(), "A and B must be on the same device");
    TORCH_CHECK(a.dim() == 2 && b.dim() == 2, "A and B must be two-dimensional");
    TORCH_CHECK(a.scalar_type() == torch::kFloat32 &&
                b.scalar_type() == torch::kFloat32, "A and B must be float32");
    TORCH_CHECK(a.is_contiguous() && b.is_contiguous(), "A and B must be contiguous");
    TORCH_CHECK(a.size(1) == b.size(0), "Size mismatch: A columns must equal B rows");
    TORCH_CHECK(!a.requires_grad() && !b.requires_grad(), "This learning kernel is forward-only");

    const c10::cuda::CUDAGuard device_guard(a.device());
    const int64_t h = a.size(0), k = a.size(1), w = b.size(1);
    auto output = torch::empty({h, w}, a.options());
    if (h == 0 || w == 0) return output;  // An empty grid cannot be launched.

    const dim3 tpb(16, 16);
    const int64_t blocks_x = (w + tpb.x - 1) / tpb.x;
    const int64_t blocks_y = (h + tpb.y - 1) / tpb.y;
    TORCH_CHECK(blocks_x <= 2147483647 && blocks_y <= 65535,
                "Output is too large for this simple 2-D launch grid");
    const dim3 blocks(blocks_x, blocks_y);
    const auto stream = c10::cuda::getCurrentCUDAStream(a.get_device());
    matmul_kernel_cuda<<<blocks, tpb, 0, stream.stream()>>>(
        a.data_ptr<float>(), b.data_ptr<float>(), output.data_ptr<float>(), h, k, w);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return output;
}
"""

In [97]:
import os
from torch.utils.cpp_extension import load_inline

assert torch.cuda.is_available(), "Run this cell in the NGC GPU kernel."
# Limit compiler memory use; restore the existing environment afterward.
_previous_max_jobs = os.environ.get("MAX_JOBS")
os.environ["MAX_JOBS"] = "1"
try:
    matmul_module = load_inline(
        name="lecture003_naive_matmul",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=["matmul_cuda"],
        extra_cflags=["-O3"],
        extra_cuda_cflags=["-O3"],
        with_cuda=True,
        verbose=False,
    )
finally:
    if _previous_max_jobs is None:
        os.environ.pop("MAX_JOBS", None)
    else:
        os.environ["MAX_JOBS"] = _previous_max_jobs

print(f"Loaded inline extension | PyTorch {torch.__version__} | {torch.cuda.get_device_name()}")
# Compilation is intentionally separate from the benchmark.

Loaded inline extension | PyTorch 2.4.0a0+3bcc3cddb5.nv24.07 | NVIDIA L4


### 8B. Check the GPU translation before timing it

Tests cover the original example, rectangular partial blocks, longer dot products,
zero-sized dimensions, invalid inputs, and a non-default stream. TF32 is disabled
for the PyTorch reference so the comparison uses float32 arithmetic. Tiny differences
remain possible because compiled multiply-add and accumulation order can differ.
The previous TF32 setting is restored after each validation or benchmark run.

In [98]:
def validate_cuda_matmul():
    device = torch.device("cuda", torch.cuda.current_device())
    previous_tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    try:
        example_a = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)
        example_b = torch.tensor([[1., 2., 3., 4.], [5., 6., 7., 8.],
                                  [9., 10., 11., 12.]], device=device)
        expected = torch.tensor([[38., 44., 50., 56.], [83., 98., 113., 128.]], device=device)
        torch.testing.assert_close(matmul_module.matmul_cuda(example_a, example_b), expected,
                                   rtol=0, atol=0)
        print("PASS: original 2x4 output")

        generator = torch.Generator(device=device).manual_seed(7)
        for h, k, w in [(17, 3, 35), (35, 19, 17), (1, 257, 1),
                         (129, 257, 65), (257, 129, 513),
                         (0, 3, 4), (2, 3, 0), (2, 0, 4)]:
            a_gpu = torch.randn(h, k, device=device, generator=generator)
            b_gpu = torch.randn(k, w, device=device, generator=generator)
            actual = matmul_module.matmul_cuda(a_gpu, b_gpu)
            reference = a_gpu @ b_gpu
            torch.testing.assert_close(actual, reference, rtol=1e-4, atol=2e-4)
            assert actual.shape == (h, w) and actual.dtype == a_gpu.dtype and actual.device == device
            print(f"PASS: ({h}, {k}) @ ({k}, {w})")

        def expect_error(label, a_input, b_input, message):
            try:
                matmul_module.matmul_cuda(a_input, b_input)
            except RuntimeError as exc:
                assert message in str(exc), str(exc)
                print(f"PASS: rejects {label}")
            else:
                raise AssertionError(f"Expected an error for {label}")

        expect_error("CPU inputs", example_a.cpu(), example_b.cpu(), "CUDA tensors")
        expect_error("mixed CPU/CUDA inputs", example_a.cpu(), example_b, "CUDA tensors")
        expect_error("wrong rank", example_a.flatten(), example_b, "two-dimensional")
        expect_error("float64", example_a.double(), example_b.double(), "float32")
        expect_error("mixed dtype", example_a.half(), example_b, "float32")
        expect_error("noncontiguous input", example_a.t(), example_b, "contiguous")
        expect_error("incompatible dimensions", example_a, example_b[:2], "Size mismatch")
        expect_error("autograd input", example_a.clone().requires_grad_(), example_b, "forward-only")

        # Create inputs, launch, and consume the output on the same non-default stream.
        stream = torch.cuda.Stream(device=device)
        with torch.cuda.stream(stream):
            stream_a = torch.randn(17, 19, device=device)
            stream_b = torch.randn(19, 35, device=device)
            stream_out = matmul_module.matmul_cuda(stream_a, stream_b)
            stream_error = (stream_out - stream_a @ stream_b).abs().max()
        stream.synchronize()
        assert stream_error.item() < 2e-4
        print("PASS: non-default stream")
        if torch.cuda.device_count() > 1:
            other = (device.index + 1) % torch.cuda.device_count()
            expect_error("different CUDA devices", example_a, example_b.to(f"cuda:{other}"), "same device")
        else:
            print("SKIP: different-CUDA-device test (one GPU available)")
        torch.cuda.synchronize(device)
        print("All available CUDA validation checks passed.")
    finally:
        torch.backends.cuda.matmul.allow_tf32 = previous_tf32

validate_cuda_matmul()

PASS: original 2x4 output
PASS: (17, 3) @ (3, 35)
PASS: (35, 19) @ (19, 17)
PASS: (1, 257) @ (257, 1)
PASS: (129, 257) @ (257, 65)
PASS: (257, 129) @ (129, 513)
PASS: (0, 3) @ (3, 4)
PASS: (2, 3) @ (3, 0)
PASS: (2, 0) @ (0, 4)
PASS: rejects CPU inputs
PASS: rejects mixed CPU/CUDA inputs
PASS: rejects wrong rank
PASS: rejects float64
PASS: rejects mixed dtype
PASS: rejects noncontiguous input
PASS: rejects incompatible dimensions
PASS: rejects autograd input
PASS: non-default stream
SKIP: different-CUDA-device test (one GPU available)
All available CUDA validation checks passed.


### 8C. Compare the translated kernel with PyTorch on the GPU

Both implementations receive identical float32 CUDA inputs. Compilation, random input creation,
and validation are outside timing. Each call allocates an output; allocation is **included** in
wall time. CUDA events measure the stream interval around a batch of full calls, not isolated
kernel time; for tiny kernels this interval can include gaps while Python submits work.
The synchronized wall measurement also includes host dispatch, allocation, and waiting.

Warm up both functions, alternate measurement order, and report medians and IQRs across batches.
TF32 is disabled for this baseline and restored afterward. This is a comparison with PyTorch's
float32 baseline, not a claim about its fastest Tensor Core settings. No CUDA Graphs are used.
Start with the default sizes; pass larger sizes to the helper after these checks succeed.

In [100]:
import statistics
import time

def benchmark_cuda_matmul(sizes=(64, 128, 512, 1024, 2048), repeats=7, calls_per_batch=10):
    assert repeats >= 4 and calls_per_batch >= 1
    device = torch.device("cuda", torch.cuda.current_device())
    previous_tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    results = []
    try:
        print(f"GPU: {torch.cuda.get_device_name(device)} | device={device} | float32 | TF32=False")
        print("Full calls, including output allocation; CUDA-event interval and synchronized wall time.")
        print(f"{'N':>6} {'implementation':>14} {'event ms':>12} {'event IQR%':>11} {'wall ms':>12} {'wall IQR%':>10}")
        for size in sizes:
            generator = torch.Generator(device=device).manual_seed(7 + size)
            a_gpu = torch.rand(size, size, device=device, generator=generator)
            b_gpu = torch.rand(size, size, device=device, generator=generator)
            custom = matmul_module.matmul_cuda(a_gpu, b_gpu)
            reference = torch.matmul(a_gpu, b_gpu)
            torch.testing.assert_close(custom, reference, rtol=1e-4, atol=1e-4)
            del custom, reference
            functions = {"translated": matmul_module.matmul_cuda, "PyTorch": torch.matmul}
            for fn in functions.values():
                for _ in range(5):
                    fn(a_gpu, b_gpu)
            torch.cuda.synchronize(device)
            samples = {name: {"event": [], "wall": []} for name in functions}
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            # Materialize event objects before measuring wall time.
            start_event.record(); end_event.record(); end_event.synchronize()
            for repeat in range(repeats):
                order = list(functions) if repeat % 2 == 0 else list(reversed(functions))
                for name in order:
                    fn = functions[name]
                    torch.cuda.synchronize(device)
                    wall_start = time.perf_counter()
                    start_event.record()
                    for _ in range(calls_per_batch):
                        output = fn(a_gpu, b_gpu)
                        del output
                    end_event.record()
                    end_event.synchronize()
                    samples[name]["wall"].append((time.perf_counter() - wall_start) * 1000 / calls_per_batch)
                    samples[name]["event"].append(start_event.elapsed_time(end_event) / calls_per_batch)
            for name, measured in samples.items():
                event_median = statistics.median(measured["event"])
                wall_median = statistics.median(measured["wall"])
                event_q1, _, event_q3 = statistics.quantiles(measured["event"], n=4, method="inclusive")
                wall_q1, _, wall_q3 = statistics.quantiles(measured["wall"], n=4, method="inclusive")
                event_iqr = 100 * (event_q3 - event_q1) / event_median
                wall_iqr = 100 * (wall_q3 - wall_q1) / wall_median
                print(f"{size:6d} {name:>14} {event_median:12.4f} {event_iqr:11.1f} {wall_median:12.4f} {wall_iqr:10.1f}")
                results.append(dict(size=size, implementation=name, event_ms=event_median,
                                    wall_ms=wall_median, event_iqr_percent=event_iqr,
                                    wall_iqr_percent=wall_iqr, samples=measured))
    finally:
        torch.backends.cuda.matmul.allow_tf32 = previous_tf32
    return results

cuda_results = benchmark_cuda_matmul()
# Optional larger run after the default sizes pass:
cuda_large_results = benchmark_cuda_matmul(sizes=(4096, 8192), calls_per_batch=3)

GPU: NVIDIA L4 | device=cuda:0 | float32 | TF32=False
Full calls, including output allocation; CUDA-event interval and synchronized wall time.
     N implementation     event ms  event IQR%      wall ms  wall IQR%
    64     translated       0.0171        37.7       0.0207       40.6
    64        PyTorch       0.0390        33.7       0.0442       32.1
   128     translated       0.0098         9.9       0.0118        9.8
   128        PyTorch       0.0206        15.2       0.0223       14.1
   512     translated       0.1605         0.3       0.1634        1.0
   512        PyTorch       0.0330         3.3       0.0356        5.7
  1024     translated       1.6030         5.0       1.6069        5.0
  1024        PyTorch       0.1795         5.6       0.1822        6.0
  2048     translated      12.8946         3.0      12.9053        3.0
  2048        PyTorch       1.1725         1.5       1.1805        1.4
GPU: NVIDIA L4 | device=cuda:0 | float32 | TF32=False
Full calls, including 

### What changed, and what to learn next

The arithmetic is still your original dot-product loop. GPU execution removes the Python
scalar-operation overhead and lets many threads calculate different output elements at once.
The naive kernel still loads A and B repeatedly across threads (caches may satisfy some loads),
and it does not explicitly stage shared tiles or use Tensor Core matrix instructions.
Optimized matmul routines organize data reuse and choose kernels for the hardware and shapes.

Before adding optimizations, trace thread (x=3, y=0) in block (x=1, y=1):
which output element does it own, which offsets does it read at i=0, and how do those
reads change at i=1? Explain why no synchronization between threads is needed in this kernel.

## Exercise ledger

Updated 2026-09-09 from the tutoring conversation and saved notebook. “Checks recorded” means the saved cell has enabled assertions, an execution count, and no recorded error; assertions were not rerun for this ledger update. Completion does not establish independent mastery. Attempt counts are minimum observed submissions, not execution counts; unknown counts and unassessed scores are left explicit.

Ledger mapping: E1–E6 match exercises 1–6; E7 is exercise 6B; E8 is exercise 7; E9 is exercise 8.

| Item | Skill targets | Status | Attempts | Hints | Score | Gap tags | Next drill |
|---|---|---|---|---|---|---|---|
| E1 | tensor-layout, row-major-indexing | Implemented; check recorded | Unknown | Not observed | Not assessed | Retention unassessed | Explain the three channel offsets for a new pixel index. |
| E2 | loop-construction, tensor-layout | Completed with guidance; checks recorded | At least 2 | Return and reshape guidance | Not assessed | explicit-return, output-shape | Recreate the grayscale loop and explain its return shape without notes. |
| E3 | loop-construction, grid-block-thread | Implemented; check recorded | Unknown | Not observed | Not assessed | Independent explanation unassessed | Explain the difference between one kernel invocation and the launcher loop. |
| E4 | bounds-check, grid-block-thread | Implemented; launch-math checks recorded; explanation present | Unknown | Not observed | Not assessed | partial-block reasoning; guard not tested by saved assertions | Explain why whole blocks create excess threads and trace valid/invalid indices in the final block. |
| E5 | loop-construction, matmul-work-count | Completed with substantial guidance; checks recorded; independent retry due | At least 1 | Repeated verbal walkthroughs and interactive visual | Not assessed | loop-nesting, dot-product-accumulation, output-store | Trace one cell, then the next column and next row; reconstruct the three loops without notes. |
| E6 | broadcasting-shapes | Completed with substantial guidance; checks recorded; independent retry due | At least 3 | Slicing, singleton dimensions, operand choice, reduction axis | Not assessed | integer-vs-slice, singleton-axis, elementwise-broadcasting, reduction-axis | Predict every intermediate shape and explain why reducing dimension zero leaves one value per column. |
| E7 | broadcasting-shapes, loop-construction | Completed with substantial guidance; check and performance explanation recorded; independent retry due | At least 2 | Loop bound, output assembly, return, SIMD/compiled-operation explanation, labeled prints | Not assessed | row-iteration, broadcast-products, reduction-axis, explicit-return | Explain one full iteration with prints hidden, then rebuild the function and explain reduced Python overhead. |
| E8 | grid-block-thread, row-major-indexing | Indexing implemented; checks recorded; tile explanation outstanding | Unknown | Not observed in this conversation | Not assessed | explicit-tile-description; independent reasoning unassessed | State the rows and columns owned by block (2,1), then derive a different tile and flat offsets. |
| E9 | cuda-host-device, shared-memory-tiling | Pending; placeholder only | 0 observed | None observed | Not assessed | Not yet assessed | Attempt the wrapper/kernel challenge and explain two reasons naive matmul loses to optimized matmul. |

### Session evidence and follow-up

- **E1:** Saved solution uses spatial size, flattening, and channel offsets; the one-pixel assertion has a recorded execution without an error. No tutoring attempt history was observed.
- **E2:** Initial function filled its output but returned `None`. Guidance identified the missing return and restoration of the image shape. Saved solution includes the return and both assertions.
- **E3:** Saved implementation separates one-index grayscale work from the serial launcher and has an enabled, executed assertion. Independent explanation was not assessed in this conversation.
- **E4:** Saved index/ceiling-division assertions executed without a recorded error; a bounds guard and learner explanation are present. The explanation recognizes excess threads, but should explicitly connect them to allocating whole blocks. The guard itself is not covered by those saved assertions.
- **E5:** Learner initially guessed code and reported not understanding it. Correctly identified matching entries and products through Socratic prompts. Guidance traced fixed output indices, movement along the shared dimension, accumulation, storing a cell, and progression across columns/rows. Learner reported improved understanding; independent reconstruction and work-count reasoning remain unassessed.
- **E6:** Initial attempt multiplied the full A by B. The next attempt used the reshaped row correctly but summed dimension one. The saved solution uses the intended operands and dimension zero, with all shape/value assertions enabled and a recorded execution. Tutoring covered colon slicing, integer indices removing axes, `None` adding an axis, conceptual broadcasting across B's rows, and the axis removed by reduction. The prior ledger's initial worked-shape hint is included in this qualitative history; an exact total hint count is unavailable.
- **E7 / 6B:** Learner first proposed iterating over the tensor rather than its row count, then correctly selected the row count. Submitted a correct broadcast loop without a return; guidance identified the missing return. Learner connected speed to SIMD but remained unsure about multiplication/reduction semantics. Explanation clarified reduced Python scalar-operation overhead and compiled tensor operations. Learner added diagnostic prints; assistant labeled four prints. Saved notebook includes the return, recorded assertion execution, and a written performance explanation. Guided correctness is established by the saved evidence; independent mastery is not.
- **E8 / exercise 7:** Later saved work contains the 2-D index and flat-offset expressions with all three assertions enabled and executed without a recorded error. This was not discussed in the tutoring context. The requested verbal output-tile description is not present.
- **E9 / exercise 8:** No implementation or performance explanation is present.

### Learning preference

Use natural-language explanations and small visual traces first. Do not supply solution code unless requested. Ask the learner to explain an intermediate value or shape before proceeding; use an independent retry to assess retention rather than treating a guided pass as mastery.
